# Feature Extraction using TSFRESH

Imports

In [10]:
import pandas as pd
import numpy as np
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from sklearn.preprocessing import LabelEncoder

In [11]:
df = pd.read_parquet("cleaned_data.parquet")

# Ensure date column is datetime
df['order_date'] = pd.to_datetime(df['order_date'])
print("Data loaded and date column converted to datetime.")
print (df.head())


Data loaded and date column converted to datetime.
     Industry    Product_Family     CHF  fill_rate order_date  Customer_id  \
0  Automotive  TF Forming Tools   25.61   0.034615 2023-01-02            1   
1  Automotive  TF Forming Tools   51.84   0.014286 2023-01-02            1   
2  Automotive  TF Forming Tools   55.65   0.090476 2023-01-02            1   
3  Automotive  TF Forming Tools   33.53   0.050000 2023-01-02            1   
4  Automotive  TF Cutting Tools  103.35   0.050000 2023-01-02            1   

   coating_id weekday  
0           1  Monday  
1           1  Monday  
2           2  Monday  
3           2  Monday  
4           3  Monday  


Customer and order level features

In [12]:
# Group by day & coating for stats
daily_stats = df.groupby(['order_date', 'coating_id']).agg(
    fill_rate_sum=('fill_rate', 'sum'),
    fill_rate_mean=('fill_rate', 'mean'),
    fill_rate_std=('fill_rate', 'std'),
    fill_rate_min=('fill_rate', 'min'),
    fill_rate_max=('fill_rate', 'max'),

    CHF_sum=('CHF', 'sum'),
    CHF_mean=('CHF', 'mean'),
    CHF_std=('CHF', 'std'),
    CHF_min=('CHF', 'min'),
    CHF_max=('CHF', 'max'),

    n_orders=('Customer_id', 'count'),
    n_unique_customers=('Customer_id', 'nunique'),
).reset_index()

# Fill NaN (from std on single order days) with 0
for col in daily_stats.columns:
    if daily_stats[col].dtype == 'float':
        daily_stats[col] = daily_stats[col].fillna(0)


In [13]:
daily_stats['weekday'] = daily_stats['order_date'].dt.day_name()
daily_stats['month'] = daily_stats['order_date'].dt.month

# Encode weekday as numbers for ML
le = LabelEncoder()
daily_stats['weekday_num'] = le.fit_transform(daily_stats['weekday'])


In [14]:
prod_ind = df.groupby(['order_date', 'coating_id']).agg({
    'Product_Family': lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown',
    'Industry': lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown',
}).reset_index()

# Optionally encode for ML (not needed for TSFRESH)
prod_ind['Product_Family_enc'] = le.fit_transform(prod_ind['Product_Family'])
prod_ind['Industry_enc'] = le.fit_transform(prod_ind['Industry'])

daily_stats = daily_stats.merge(prod_ind, on=['order_date', 'coating_id'], how='left')


Rolling/Lag features

In [15]:
daily_stats = daily_stats.sort_values(['coating_id', 'order_date'])

for lag in [1, 2, 3, 7, 14]:
    daily_stats[f'fill_rate_sum_lag{lag}'] = daily_stats.groupby('coating_id')['fill_rate_sum'].shift(lag)
    daily_stats[f'fill_rate_mean_lag{lag}'] = daily_stats.groupby('coating_id')['fill_rate_mean'].shift(lag)
    daily_stats[f'CHF_sum_lag{lag}'] = daily_stats.groupby('coating_id')['CHF_sum'].shift(lag)
    # You can do this for any feature you want!

# Rolling mean/volatility
for window in [3, 7, 14]:
    daily_stats[f'fill_rate_sum_rollmean{window}'] = daily_stats.groupby('coating_id')['fill_rate_sum'].rolling(window).mean().reset_index(0, drop=True)
    daily_stats[f'fill_rate_sum_rollstd{window}'] = daily_stats.groupby('coating_id')['fill_rate_sum'].rolling(window).std().reset_index(0, drop=True)


TSFresh

In [16]:
# TSFRESH expects a long format: id, time, value, plus any others
tsfresh_input = daily_stats.rename(columns={
    'coating_id': 'id',
    'order_date': 'time'
})

# We'll use fill_rate_sum and CHF_sum as two separate features
# Prepare in long format for multi-variate extraction
melted = pd.melt(tsfresh_input,
                 id_vars=['id', 'time'],
                 value_vars=['fill_rate_sum', 'CHF_sum'],
                 var_name='kind',
                 value_name='value')

# Now pivot to wide format for TSFRESH's "column_kind" argument
pivoted = melted.pivot_table(index=['id', 'time'], columns='kind', values='value').reset_index()

# Extract features
tsfresh_feats = extract_features(
    pivoted,
    column_id='id',
    column_sort='time',
    column_kind=None,    # Uses all columns except id/time as "kinds"
    column_value=None,   # All other columns are values
    default_fc_parameters=None,  # None=all possible features (very deep)
    n_jobs=4
)

# Impute missing values from TSFRESH (required)
impute(tsfresh_feats)

Feature Extraction: 100%|██████████| 18/18 [00:09<00:00,  1.95it/s]
c:\Users\lars-\Backup Desktop\03 München\01 Kurse TUM\04 Modeling, Optimization and Simulation in Operations Management\03 Code\env_mos\lib\site-packages\tsfresh\utilities\dataframe_functions.py:198: RuntimeWarning: The columns ['CHF_sum__query_similarity_count__query_None__threshold_0.0'
 'fill_rate_sum__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


,CHF_sum__variance_larger_than_standard_deviation,CHF_sum__has_duplicate_max,CHF_sum__has_duplicate_min,CHF_sum__has_duplicate,CHF_sum__sum_values,CHF_sum__abs_energy,CHF_sum__mean_abs_change,CHF_sum__mean_change,CHF_sum__mean_second_derivative_central,CHF_sum__median,...,fill_rate_sum__fourier_entropy__bins_5,fill_rate_sum__fourier_entropy__bins_10,fill_rate_sum__fourier_entropy__bins_100,fill_rate_sum__permutation_entropy__dimension_3__tau_1,fill_rate_sum__permutation_entropy__dimension_4__tau_1,fill_rate_sum__permutation_entropy__dimension_5__tau_1,fill_rate_sum__permutation_entropy__dimension_6__tau_1,fill_rate_sum__permutation_entropy__dimension_7__tau_1,fill_rate_sum__query_similarity_count__query_None__threshold_0.0,fill_rate_sum__mean_n_absolute_max__number_of_maxima_7
1,1.0,0.0,0.0,0.0,2885753.85,2.680631e+10,4615.890202,25.593585,13.298140,4074.640,...,1.024540,1.672364,3.666673,1.787777,3.158792,4.669500,5.861775,6.223475,0.0,395.567894
2,1.0,0.0,0.0,1.0,649071.93,1.634394e+09,1217.603419,7.336912,2.833029,755.770,...,1.402821,2.057191,3.949376,1.787124,3.156788,4.681370,5.833486,6.227988,0.0,119.277808
3,1.0,0.0,1.0,1.0,27659210.10,2.008332e+12,31453.096915,232.096588,94.273255,42063.705,...,1.295931,1.947021,3.910983,1.789514,3.159893,4.671796,5.840512,6.221993,0.0,395.911632
4,1.0,0.0,0.0,0.0,1466617.33,1.475220e+10,3615.946366,13.571239,4.769779,1801.820,...,1.176827,1.849635,3.769132,1.787520,3.146147,4.638487,5.737584,6.111430,0.0,25.178041
5,1.0,0.0,0.0,0.0,2161263.74,1.808015e+10,3844.648598,25.730953,17.590787,2982.875,...,1.074948,1.700193,3.667248,1.789230,3.160464,4.684982,5.849570,6.194061,0.0,10.777739
6,1.0,0.0,1.0,1.0,6675683.90,1.186214e+11,9112.500037,13.239703,5.579991,10835.860,...,1.504074,2.182595,4.142362,1.787320,3.156831,4.676846,5.836661,6.197893,0.0,857.842364
7,1.0,0.0,0.0,0.0,2739063.73,4.323798e+10,5626.598096,46.553697,21.266028,3064.860,...,1.341531,2.011642,3.959811,1.788636,3.157863,4.665959,5.818244,6.195221,0.0,199.707950
8,1.0,0.0,0.0,0.0,1570412.40,9.664649e+09,3195.384202,21.903284,12.127978,1767.380,...,1.427633,2.041706,3.963053,1.786210,3.147714,4.664321,5.833364,6.196582,0.0,123.705063
9,1.0,0.0,1.0,1.0,4315670.29,1.109220e+11,9931.800220,66.763187,30.334679,3722.230,...,0.916283,1.568600,3.626599,1.787993,3.145836,4.638130,5.775717,6.192516,0.0,97.751131
10,1.0,0.0,0.0,0.0,874675.74,3.506645e+09,1905.984135,5.441730,5.852273,1219.530,...,1.352744,1.998474,3.973589,1.782017,3.148117,4.616630,5.741125,6.051944,0.0,13.423826


Aggregate data

In [17]:
# Attach target: next day’s fill_rate_sum per coating_id
daily_stats = daily_stats.sort_values(['coating_id', 'order_date'])
daily_stats['target_fill_rate_sum'] = daily_stats.groupby('coating_id')['fill_rate_sum'].shift(-1)
print(daily_stats)

# Drop rows with missing target (cannot predict last day)
model_data = daily_stats.dropna(subset=['target_fill_rate_sum'])

# Merge TSFRESH features (on coating_id)
model_data = model_data.merge(tsfresh_feats, left_on='coating_id', right_index=True, how='left')

# Optional: drop columns not needed for ML
drop_cols = ['order_date', 'fill_rate_sum', 'target_fill_rate_sum']
X = model_data.drop(columns=drop_cols)
y = model_data['target_fill_rate_sum']


      order_date  coating_id  fill_rate_sum  fill_rate_mean  fill_rate_std  \
0     2023-01-02           1       0.610664        0.011522       0.015353   
26    2023-01-03           1      35.836608        0.833409       4.695023   
56    2023-01-04           1       9.581249        0.107654       0.885597   
83    2023-01-05           1      27.812082        0.386279       2.822320   
114   2023-01-06           1       3.867537        0.120861       0.303700   
...          ...         ...            ...             ...            ...   
16755 2025-03-25         121       0.443099        0.055387       0.075806   
16784 2025-03-26         121       3.214789        0.107160       0.201089   
16813 2025-03-27         121       2.881788        0.106733       0.136873   
16840 2025-03-28         121       3.410087        0.048716       0.076893   
16874 2025-03-31         121      81.787920        0.228458       0.605928   

       fill_rate_min  fill_rate_max   CHF_sum    CHF_mean     C